# Weather Ensemble — Database Explorer

Explore `data/weather.db` directly with SQL. Run the cells top to bottom, then use the last cell as a scratchpad for your own queries.

Tables:
- `forecasts` — every provider's forecast for a given date, as collected
- `actuals` — the real observed weather Open-Meteo recorded for that date
- `ensemble_predictions` — the weighted-average blended forecasts this project has generated

Select the **Weather Ensemble (.venv)** kernel before running.

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

DB_PATH = Path("../data/weather.db") if Path("../data/weather.db").exists() else Path("data/weather.db")
conn = sqlite3.connect(DB_PATH)


def q(sql: str, params: tuple = ()) -> pd.DataFrame:
    """Run a raw SQL query against the weather database and return a DataFrame."""
    return pd.read_sql_query(sql, conn, params=params)


print(f"Connected to {DB_PATH.resolve()}")

Connected to C:\Users\zephyr.howson\Desktop\weather-ensemble-forecaster\data\weather.db


## Schema — what tables and columns exist

In [2]:
q("SELECT name, type FROM sqlite_master WHERE type='table' ORDER BY name")

,name,type
0,actuals,table
1,ensemble_predictions,table
2,forecasts,table
3,sqlite_sequence,table


In [3]:
for table in ["forecasts", "actuals", "ensemble_predictions"]:
    print(f"\n--- {table} ---")
    display(q(f"PRAGMA table_info({table})")[["name", "type"]])


--- forecasts ---


,name,type
0,id,INTEGER
1,source,TEXT
2,location_name,TEXT
3,lat,REAL
4,lon,REAL
5,forecast_date,TEXT
6,collected_at,TEXT
7,max_temp,REAL
8,min_temp,REAL
9,rain_probability,REAL



--- actuals ---


,name,type
0,id,INTEGER
1,source,TEXT
2,location_name,TEXT
3,lat,REAL
4,lon,REAL
5,actual_date,TEXT
6,collected_at,TEXT
7,max_temp,REAL
8,min_temp,REAL
9,precipitation_sum,REAL



--- ensemble_predictions ---


,name,type
0,id,INTEGER
1,location_name,TEXT
2,lat,REAL
3,lon,REAL
4,forecast_date,TEXT
5,generated_at,TEXT
6,window_days,INTEGER
7,max_temp,REAL
8,min_temp,REAL
9,rain_probability,REAL


## Row counts — how much data do we actually have?

In [4]:
q("""
SELECT 'forecasts' AS table_name, COUNT(*) AS rows FROM forecasts
UNION ALL
SELECT 'actuals', COUNT(*) FROM actuals
UNION ALL
SELECT 'ensemble_predictions', COUNT(*) FROM ensemble_predictions
""")

,table_name,rows
0,forecasts,864
1,actuals,393
2,ensemble_predictions,3


In [5]:
q("""
SELECT source, location_name, COUNT(*) AS forecast_rows,
       MIN(forecast_date) AS earliest, MAX(forecast_date) AS latest
FROM forecasts
GROUP BY source, location_name
ORDER BY location_name, source
""")

,source,location_name,forecast_rows,earliest,latest
0,open_meteo_best_match,Melbourne,5,2026-06-06,2026-07-04
1,open_meteo_ecmwf_ifs025,Melbourne,368,2025-06-05,2026-07-04
2,open_meteo_gem_seamless,Melbourne,91,2026-04-04,2026-07-04
3,open_meteo_gfs_global,Melbourne,396,2025-06-05,2026-07-04
4,wttr_in,Melbourne,4,2026-06-06,2026-07-04


## Latest forecast per source, for a given location

Change `LOCATION` below to match whatever `--name` you've been running the CLI with.

In [6]:
LOCATION = "Melbourne"

q("""
SELECT f.source, f.forecast_date, f.collected_at,
       f.max_temp, f.min_temp, f.rain_probability, f.precipitation_sum,
       f.wind_speed, f.uv_index
FROM forecasts f
JOIN (
    SELECT source, MAX(collected_at) AS latest_collected_at
    FROM forecasts
    WHERE location_name = ?
    GROUP BY source
) latest ON f.source = latest.source AND f.collected_at = latest.latest_collected_at
WHERE f.location_name = ?
ORDER BY f.source
""", params=(LOCATION, LOCATION))

,source,forecast_date,collected_at,max_temp,min_temp,rain_probability,precipitation_sum,wind_speed,uv_index
0,open_meteo_best_match,2026-07-04,2026-07-03T13:07:49,12.9,9.1,92.0,1.3,13.1,1.3
1,open_meteo_ecmwf_ifs025,2026-07-04,2026-07-03T13:07:51,13.0,8.8,92.0,0.0,13.0,NaN
2,open_meteo_gem_seamless,2026-07-04,2026-07-03T13:07:54,13.0,8.4,62.0,1.8,16.2,NaN
3,open_meteo_gfs_global,2026-07-04,2026-07-03T13:07:52,12.0,9.6,7.0,0.6,15.0,1.3
4,wttr_in,2026-07-04,2026-07-03T13:07:56,12.0,9.0,24.0,NaN,16.0,1.0


## Recent actual weather

In [7]:
q("""
SELECT actual_date, max_temp, min_temp, precipitation_sum, did_rain, wind_speed, uv_index
FROM actuals
WHERE location_name = ?
ORDER BY actual_date DESC
LIMIT 14
""", params=(LOCATION,))

,actual_date,max_temp,min_temp,precipitation_sum,did_rain,wind_speed,uv_index
0,2026-07-02,12.9,10.0,9.8,1,27.9,None
1,2026-07-01,15.9,11.4,4.2,1,23.7,None
2,2026-06-30,14.9,12.0,17.1,1,24.6,None
3,2026-06-29,15.8,7.5,0.3,1,19.9,None
4,2026-06-28,14.2,4.9,0.0,0,6.2,None
5,2026-06-27,11.4,6.5,0.0,0,4.7,None
6,2026-06-26,13.3,7.1,0.0,0,6.4,None
7,2026-06-25,12.9,7.4,0.0,0,7.5,None
8,2026-06-24,15.5,8.9,0.6,1,9.5,None
9,2026-06-23,15.7,6.5,0.1,0,9.0,None


## Forecast vs. actual, joined

This is the same join `compute_mae_scores` runs internally — it's the table the accuracy weighting is built on.

In [8]:
modelling = q("""
SELECT f.source, f.forecast_date,
       f.max_temp, a.max_temp AS actual_max_temp,
       f.min_temp, a.min_temp AS actual_min_temp,
       f.precipitation_sum, a.precipitation_sum AS actual_precipitation_sum
FROM forecasts f
JOIN actuals a
  ON f.location_name = a.location_name AND f.forecast_date = a.actual_date
WHERE f.location_name = ?
ORDER BY f.forecast_date DESC, f.source
""", params=(LOCATION,))
modelling.head(20)

,source,forecast_date,max_temp,actual_max_temp,min_temp,actual_min_temp,precipitation_sum,actual_precipitation_sum
0,open_meteo_gem_seamless,2026-07-02,12.1,12.9,9.2,10.0,4.6,9.8
1,open_meteo_gfs_global,2026-07-02,12.6,12.9,10.5,10.0,2.9,9.8
2,open_meteo_gem_seamless,2026-07-01,15.1,15.9,11.3,11.4,6.2,4.2
3,open_meteo_gfs_global,2026-07-01,17.2,15.9,11.7,11.4,2.0,4.2
4,open_meteo_gem_seamless,2026-06-30,14.8,14.9,11.6,12.0,10.4,17.1
5,open_meteo_gfs_global,2026-06-30,14.4,14.9,12.5,12.0,4.9,17.1
6,open_meteo_gem_seamless,2026-06-29,16.2,15.8,7.6,7.5,0.9,0.3
7,open_meteo_gfs_global,2026-06-29,15.4,15.8,11.5,7.5,1.3,0.3
8,open_meteo_gem_seamless,2026-06-28,15.9,14.2,3.8,4.9,0.0,0.0
9,open_meteo_gfs_global,2026-06-28,17.1,14.2,6.4,4.9,0.0,0.0


## Mean absolute error per source

These are the numbers that drive each source's weight in `blend_forecast` — lower MAE means a higher `1/MAE` weight.

In [9]:
mae = modelling.copy()
mae["abs_err_max_temp"] = (mae["max_temp"] - mae["actual_max_temp"]).abs()
mae["abs_err_min_temp"] = (mae["min_temp"] - mae["actual_min_temp"]).abs()
mae["abs_err_precip"] = (mae["precipitation_sum"] - mae["actual_precipitation_sum"]).abs()

mae.groupby("source")[["abs_err_max_temp", "abs_err_min_temp", "abs_err_precip"]].mean().round(3)

,abs_err_max_temp,abs_err_min_temp,abs_err_precip
source,,,
open_meteo_best_match,0.225,0.575,0.325
open_meteo_ecmwf_ifs025,0.541,0.429,0.912
open_meteo_gem_seamless,0.861,0.977,1.479
open_meteo_gfs_global,0.981,1.092,1.381
wttr_in,0.000,1.200,NaN


## Blended forecasts already generated

In [ ]:
q("""
SELECT forecast_date, generated_at, window_days, max_temp, min_temp,
       rain_probability, precipitation_sum, did_rain, wind_speed
FROM ensemble_predictions
WHERE location_name = ?
ORDER BY generated_at DESC
LIMIT 10
""", params=(LOCATION,))

## Your own SQL — scratchpad

Use `q("...")` with any SQL against `forecasts`, `actuals`, or `ensemble_predictions`.

In [20]:
q("""
SELECT *
FROM forecasts
WHERE source='wttr_in'
ORDER BY forecast_date DESC
""")

,id,source,location_name,lat,lon,forecast_date,collected_at,max_temp,min_temp,rain_probability,precipitation_sum,uv_index,wind_speed,wind_gusts,cloud_cover,humidity,pressure_msl,weather_code,raw_json,collection_method
0,3511,wttr_in,Melbourne,-37.8136,144.9631,2026-07-04,2026-07-03T16:09:21,12.0,9.0,24.0,None,1.0,16.0,25.0,81.875,69.625,1029.125,176.0,"{""current_condition"": [{""FeelsLikeC"": ""11"", ""F...",live
1,3512,wttr_in,Melbourne,-37.8136,144.9631,2026-07-04,2026-07-03T16:10:43,12.0,9.0,24.0,None,1.0,16.0,25.0,81.875,69.625,1029.125,176.0,"{""current_condition"": [{""FeelsLikeC"": ""11"", ""F...",live
2,3513,wttr_in,Melbourne,-37.8136,144.9631,2026-07-04,2026-07-03T16:14:43,12.0,9.0,24.0,None,1.0,16.0,25.0,81.875,69.625,1029.125,176.0,"{""current_condition"": [{""FeelsLikeC"": ""11"", ""F...",live


In [18]:
q("""
SELECT *
FROM ensemble_predictions
""")

,id,location_name,lat,lon,forecast_date,generated_at,window_days,max_temp,min_temp,rain_probability,precipitation_sum,did_rain,uv_index,wind_speed,wind_gusts,cloud_cover,humidity,pressure_msl,weather_code,metadata_json
